<a href="https://colab.research.google.com/github/isaac-sun/fedfree/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# FedFree — Federated Learning Free-Rider Defense

**DistilBERT + PEFT LoRA + Per-Class Shapley Detection on Yahoo Answers**

Two-phase defense: Positive-Sum Filter → Variance Fingerprinting

## 1. Setup

Clone the repository.

In [ ]:
import os, sys

REPO_NAME = "fedfree"
GITHUB_USER = "isaac-sun"

if os.path.basename(os.getcwd()) == REPO_NAME:
    print("Already in fedfree/ — pulling latest...")
    !git pull origin main
elif os.path.exists(REPO_NAME):
    %cd {REPO_NAME}
    !git pull origin main
else:
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_NAME}

print(f"Working directory: {os.getcwd()}")

## 2. Install Dependencies

In [ ]:
# Force compatible versions (Colab's pre-installed huggingface-hub is too new)
!pip install -q --force-reinstall 'huggingface-hub==0.25.2' 'datasets==2.21.0'
!pip install -q torch transformers peft scikit-learn numpy pandas matplotlib seaborn tqdm
print("Dependencies installed.")

## 3. Verify GPU

In [ ]:
import torch
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM:     {vram:.1f} GB")
else:
    print("Running on CPU — may be slow.")

## 4. Run Experiments

Two experiments: **Baseline** (no defense) and **Defense** (FedFree).

Expected runtime:
- **T4 GPU**: ~30–45 min
- **A100 GPU**: ~15–20 min
- **CPU**: ~2–3 hours

In [ ]:
%run main.py

## 5. Generate Visualizations

In [ ]:
%run plot_defense.py

## 6. View Results

In [ ]:
import pandas as pd
from IPython.display import Image, display as ipy_display
import os

# Show defense history
csv_path = "results/defense_history.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Records: {len(df)} | Rounds: {df['round'].nunique()} | Clients: {df['client_id'].nunique()}")
    print(f"\nDetection accuracy:")
    tp = df[(df['is_attacker']) & (df['suspected'])].shape[0]
    fn = df[(df['is_attacker']) & (~df['suspected'])].shape[0]
    tn = df[(~df['is_attacker']) & (~df['suspected'])].shape[0]
    fp = df[(~df['is_attacker']) & (df['suspected'])].shape[0]
    print(f"  True Positive (caught):  {tp}")
    print(f"  False Negative (missed): {fn}")
    print(f"  True Negative (clean):   {tn}")
    print(f"  False Positive (wrong):  {fp}")
    if tp + fn > 0:
        print(f"  Recall: {tp/(tp+fn):.1%}")
    if tp + fp > 0:
        print(f"  Precision: {tp/(tp+fp):.1%}")

# Show plots
plots_dir = "results/plots"
if os.path.isdir(plots_dir):
    for f in sorted(os.listdir(plots_dir)):
        if f.endswith(".png"):
            print(f"\n### {f}")
            ipy_display(Image(os.path.join(plots_dir, f)))

## 7. Download Results

In [ ]:
!zip -r fedfree_results.zip results/ 2>/dev/null
try:
    from google.colab import files
    files.download("fedfree_results.zip")
except ImportError:
    print("fedfree_results.zip ready for manual download.")